In [2]:
import os
import numpy as np
import pandas as pd
from datetime import datetime

from math import radians, sin, cos, sqrt, atan2

In [3]:
# Navigate to your project
os.chdir('/workspaces/DSE3101-Project')

# Verify you're in the right place
print("Current directory:", os.getcwd())
print("Contents:", os.listdir('.'))

# Navigate to data/raw folder
os.chdir('data/raw')

Current directory: /workspaces/DSE3101-Project
Contents: ['docs', '.git', 'notebooks', '.DS_Store', 'models', 'frontend', 'requirements.txt', '.gitignore', '.vscode', 'backend', 'Final Web Scraper.ipynb', 'data', 'onemap_all_themes_full.json', 'README.md', 'onemap_all_themes_raw.txt']


In [4]:
df = pd.read_csv('propertyguru_full.csv')

# Check to see if it loaded correctly
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20883 entries, 0 to 20882
Data columns (total 33 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   listing_url                        20883 non-null  str    
 1   exact_address                      20880 non-null  str    
 2   price_detail                       20702 non-null  str    
 3   town_detail                        20800 non-null  str    
 4   bedrooms_detail                    20800 non-null  float64
 5   bathrooms_detail                   20161 non-null  float64
 6   area_detail                        20701 non-null  str    
 7   floor_detail                       9935 non-null   str    
 8   property_type_detail               20800 non-null  str    
 9   address_from_url                   20883 non-null  str    
 10  listing_id                         20883 non-null  int64  
 11  postal_code                        20265 non-null  float64
 12  o

In [5]:
# Count how many rows are duplicate (i.e., not the first occurrence)
num_duplicates = df.duplicated(subset=["listing_url"]).sum()
print("Number of duplicate rows:", num_duplicates)

df = df.drop_duplicates(subset=["listing_url"], keep="first")

Number of duplicate rows: 6112


In [6]:
df.shape

(14771, 33)

In [7]:
# Missing values
missing_values = df.isnull().sum()
print("Missing values per column:")
print(missing_values)

Missing values per column:
listing_url                              0
exact_address                            2
price_detail                           128
town_detail                             61
bedrooms_detail                         61
bathrooms_detail                       515
area_detail                            129
floor_detail                          7773
property_type_detail                    61
address_from_url                         0
listing_id                               0
postal_code                            616
onemap_full_address                    608
onemap_road_name                       608
latitude                               608
longitude                              608
nearest_mrt_name                       608
nearest_mrt_exit                       608
nearest_mrt_distance_m                 608
num_mrt_within_1000m                   608
nearest_clinic_name                  14623
nearest_clinic_distance_m              608
nearest_park_name          

In [8]:
critical_cols = [
    'price_detail', # important to get over/under valuation label
    'area_detail', # important to predict fair valuation price using xgboost model
    'num_amenities_within_1000m', # important to calculate SAI score
]

# Drop rows where any of the critical columns have missing values
clean_df = df.dropna(subset=critical_cols)

In [9]:
clean_df.info()

<class 'pandas.DataFrame'>
Index: 14042 entries, 0 to 20882
Data columns (total 33 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   listing_url                        14042 non-null  str    
 1   exact_address                      14042 non-null  str    
 2   price_detail                       14042 non-null  str    
 3   town_detail                        14042 non-null  str    
 4   bedrooms_detail                    14042 non-null  float64
 5   bathrooms_detail                   13664 non-null  float64
 6   area_detail                        14042 non-null  str    
 7   floor_detail                       6877 non-null   str    
 8   property_type_detail               14042 non-null  str    
 9   address_from_url                   14042 non-null  str    
 10  listing_id                         14042 non-null  int64  
 11  postal_code                        14034 non-null  float64
 12  onemap

In [10]:
clean_df['room_count_proxy'] = clean_df['bedrooms_detail'] + 1 # Living room

In [11]:
clean_df['room_count_proxy'].value_counts()

room_count_proxy
4.0     9155
3.0     2844
5.0     1753
2.0      191
6.0       76
7.0       20
1.0        2
32.0       1
Name: count, dtype: int64

In [12]:
clean_df["floor_area_sqm"] = clean_df["area_detail"].str.replace(r"\D+", "", regex=True).astype(float) * 0.092903

In [14]:
# Filter for listings with rooms <= 4 (relevant for downsizing)
mask_bedrooms = clean_df["room_count_proxy"] <= 4
# Filter for HDB listings only (relevant for downsizing)
mask_hdb = clean_df["listing_url"].str.contains("hdb", na=False, case=False)

final_df = clean_df[mask_bedrooms & mask_hdb]

In [15]:
final_df.info()

<class 'pandas.DataFrame'>
Index: 12192 entries, 0 to 20882
Data columns (total 35 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   listing_url                        12192 non-null  str    
 1   exact_address                      12192 non-null  str    
 2   price_detail                       12192 non-null  str    
 3   town_detail                        12192 non-null  str    
 4   bedrooms_detail                    12192 non-null  float64
 5   bathrooms_detail                   11823 non-null  float64
 6   area_detail                        12192 non-null  str    
 7   floor_detail                       6082 non-null   str    
 8   property_type_detail               12192 non-null  str    
 9   address_from_url                   12192 non-null  str    
 10  listing_id                         12192 non-null  int64  
 11  postal_code                        12185 non-null  float64
 12  onemap

In [16]:
# HDB Towns in Singapore:
# https://www.propertyguru.com.sg/singapore-property-listing/hdb/hdb-singapore-estates# 

station_to_town = {
    'ADMIRALTY MRT STATION': 'Woodlands',
    'ALJUNIED MRT STATION': 'Geylang',
    'ANG MO KIO MRT STATION': 'Ang Mo Kio',
    'BAKAU LRT STATION': 'Sengkang',
    'BANGKIT LRT STATION': 'Bukit Panjang',
    'BARTLEY MRT STATION': 'Serangoon',
    'BAYSHORE MRT STATION': 'Bedok',
    'BEAUTY WORLD MRT STATION': 'Bukit Timah',
    'BEDOK MRT STATION': 'Bedok',
    'BEDOK NORTH MRT STATION': 'Bedok',
    'BEDOK RESERVOIR MRT STATION': 'Bedok',
    'BENCOOLEN MRT STATION': 'Central Area',
    'BENDEMEER MRT STATION': 'Kallang/Whampoa',
    'BISHAN MRT STATION': 'Bishan',
    'BOON KENG MRT STATION': 'Kallang/Whampoa',
    'BOON LAY MRT STATION': 'Jurong West',
    'BRADDELL MRT STATION': 'Toa Payoh',
    'BRIGHT HILL MRT STATION': 'Bishan',
    'BUANGKOK MRT STATION': 'Sengkang',
    'BUGIS MRT STATION': 'Central Area',
    'BUKIT BATOK MRT STATION': 'Bukit Batok',
    'BUKIT GOMBAK MRT STATION': 'Bukit Batok',
    'BUKIT PANJANG LRT STATION': 'Bukit Panjang',
    'BUKIT PANJANG MRT STATION': 'Bukit Panjang',
    'BUONA VISTA MRT STATION': 'Queenstown',
    'CALDECOTT MRT STATION': 'Toa Payoh',
    'CANBERRA MRT STATION': 'Sembawang',
    'CC9': 'Central Area',
    'CHANGI AIRPORT MRT STATION': 'Tampines',
    'CHENG LIM LRT STATION': 'Sengkang',
    'CHINATOWN MRT STATION': 'Central Area',
    'CHINESE GARDEN MRT STATION': 'Jurong East',
    'CHOA CHU KANG MRT STATION': 'Choa Chu Kang',
    'CLEMENTI MRT STATION': 'Clementi',
    'COMMONWEALTH MRT STATION': 'Queenstown',
    'COMPASSVALE LRT STATION': 'Sengkang',
    'CORAL EDGE LRT STATION': 'Punggol',
    'COVE LRT STATION': 'Punggol',
    'DAKOTA MRT STATION': 'Geylang',
    'DAMAI LRT STATION': 'Punggol',
    'DOVER MRT STATION': 'Queenstown',
    'DT4': 'Central Area',
    'EUNOS MRT STATION': 'Geylang',
    'FAJAR LRT STATION': 'Bukit Panjang',
    'FARRER PARK MRT STATION': 'Kallang/Whampoa',
    'FARRER ROAD MRT STATION': 'Bukit Timah',
    'FARMWAY LRT STATION': 'Sengkang',
    'FERNVALE LRT STATION': 'Sengkang',
    'GEYLANG BAHRU MRT STATION': 'Kallang/Whampoa',
    'GREAT WORLD MRT STATION': 'Central Area',
    'HARBOURFRONT MRT STATION': 'Bukit Merah',
    'HAVELOCK MRT STATION': 'Bukit Merah',
    'HOLLAND VILLAGE MRT STATION': 'Queenstown',
    'HOUGANG MRT STATION': 'Hougang',
    'JALAN BESAR MRT STATION': 'Central Area',
    'JELAPANG LRT STATION': 'Bukit Panjang',
    'JURONG EAST MRT STATION': 'Jurong East',
    'KADALOOR LRT STATION': 'Punggol',
    'KAKI BUKIT MRT STATION': 'Bedok',
    'KALLANG MRT STATION': 'Kallang/Whampoa',
    'KANGKAR LRT STATION': 'Sengkang',
    'KATONG PARK MRT STATION': 'Marine Parade',
    'KEAT HONG LRT STATION': 'Choa Chu Kang',
    'KEMBANGAN MRT STATION': 'Bedok',
    'KHATIB MRT STATION': 'Yishun',
    'KOVAN MRT STATION': 'Hougang',
    'KUPANG LRT STATION': 'Sengkang',
    'LABRADOR PARK MRT STATION': 'Bukit Merah',
    'LAKESIDE MRT STATION': 'Jurong West',
    'LAVENDER MRT STATION': 'Kallang/Whampoa',
    'LAYAR LRT STATION': 'Sengkang',
    'LENTOR MRT STATION': 'Ang Mo Kio',
    'LITTLE INDIA MRT STATION': 'Central Area',
    'LORONG CHUAN MRT STATION': 'Serangoon',
    'MACPHERSON MRT STATION': 'Geylang',
    'MARINE PARADE MRT STATION': 'Marine Parade',
    'MARINE TERRACE MRT STATION': 'Marine Parade',
    'MARSILING MRT STATION': 'Woodlands',
    'MARYMOUNT MRT STATION': 'Bishan',
    'MATTAR MRT STATION': 'Geylang',
    'MAXWELL MRT STATION': 'Central Area',
    'MAYFLOWER MRT STATION': 'Ang Mo Kio',
    'MERIDIAN LRT STATION': 'Punggol',
    'MOUNTBATTEN MRT STATION': 'Geylang',
    'NIBONG LRT STATION': 'Punggol',
    'NICOLL HIGHWAY MRT STATION': 'Central Area',
    'NOVENA MRT STATION': 'Kallang/Whampoa',
    'OASIS LRT STATION': 'Punggol',
    'ONE-NORTH MRT STATION': 'Queenstown',
    'OUTRAM PARK MRT STATION': 'Central Area',
    'PASIR RIS MRT STATION': 'Pasir Ris',
    'PAYA LEBAR MRT STATION': 'Geylang',
    'PENDING LRT STATION': 'Bukit Panjang',
    'PETIR LRT STATION': 'Bukit Panjang',
    'PHOENIX LRT STATION': 'Bukit Panjang',
    'PIONEER MRT STATION': 'Jurong West',
    'POTONG PASIR MRT STATION': 'Toa Payoh',
    'PUNGGOL LRT STATION': 'Punggol',
    'PUNGGOL MRT STATION': 'Punggol',
    'PUNGGOL POINT LRT STATION': 'Punggol',
    'QUEENSTOWN MRT STATION': 'Queenstown',
    'RANGGUNG LRT STATION': 'Sengkang',
    'REDHILL MRT STATION': 'Bukit Merah',
    'RENJONG LRT STATION': 'Sengkang',
    'RIVIERA LRT STATION': 'Punggol',
    'ROCHOR MRT STATION': 'Central Area',
    'RUMBIA LRT STATION': 'Sengkang',
    'SAMUDERA LRT STATION': 'Punggol',
    'SEGAR LRT STATION': 'Bukit Panjang',
    'SEMBAWANG MRT STATION': 'Sembawang',
    'SENGKANG MRT STATION': 'Sengkang',
    'SENJA LRT STATION': 'Bukit Panjang',
    'SERANGOON MRT STATION': 'Serangoon',
    'SIMEI MRT STATION': 'Tampines',
    'SOO TECK LRT STATION': 'Punggol',
    'SOUTH VIEW LRT STATION': 'Choa Chu Kang',
    'SUMANG LRT STATION': 'Punggol',
    'TAI SENG MRT STATION': 'Geylang',
    'TAMPINES EAST MRT STATION': 'Tampines',
    'TAMPINES MRT STATION': 'Tampines',
    'TAMPINES WEST MRT STATION': 'Tampines',
    'TANAH MERAH MRT STATION': 'Bedok',
    'TANJONG PAGAR MRT STATION': 'Central Area',
    'TECK WHYE LRT STATION': 'Choa Chu Kang',
    'TELOK BLANGAH MRT STATION': 'Bukit Merah',
    'THANGGAM LRT STATION': 'Sengkang',
    'TIONG BAHRU MRT STATION': 'Bukit Merah',
    'TOA PAYOH MRT STATION': 'Toa Payoh',
    'TONGKANG LRT STATION': 'Sengkang',
    'UBI MRT STATION': 'Geylang',
    'UPPER CHANGI MRT STATION': 'Tampines',
    'UPPER THOMSON MRT STATION': 'Bishan',
    'WOODLANDS MRT STATION': 'Woodlands',
    'WOODLANDS NORTH MRT STATION': 'Woodlands',
    'WOODLANDS SOUTH MRT STATION': 'Woodlands',
    'WOODLEIGH MRT STATION': 'Toa Payoh',
    'YEW TEE MRT STATION': 'Choa Chu Kang',
    'YIO CHU KANG MRT STATION': 'Ang Mo Kio',
    'YISHUN MRT STATION': 'Yishun'
}

In [17]:
final_df = final_df.dropna(subset=['nearest_mrt_name'])
final_df['hdb_town'] = final_df['nearest_mrt_name'].map(station_to_town)

# To check if any stations were missed (print rows where the mapping failed)
missing_towns = final_df[final_df['hdb_town'].isna()]
if not missing_towns.empty:
    print("Warning: Some stations were not mapped!")
    print(missing_towns['nearest_mrt_name'].unique())
else:
    print("All stations successfully mapped to an HDB town!")

All stations successfully mapped to an HDB town!


In [18]:
final_df['hdb_town'].value_counts()

hdb_town
Sengkang           1235
Punggol             911
Tampines            814
Yishun              795
Jurong West         698
Bukit Batok         682
Woodlands           608
Ang Mo Kio          593
Bedok               585
Toa Payoh           581
Bukit Merah         518
Queenstown          500
Hougang             485
Choa Chu Kang       398
Kallang/Whampoa     374
Bukit Panjang       364
Clementi            310
Sembawang           308
Geylang             290
Bishan              241
Jurong East         234
Pasir Ris           225
Central Area        167
Serangoon           149
Marine Parade        89
Bukit Timah          38
Name: count, dtype: int64

In [19]:
region_mapping = {
    # --- CCR (Core Central Region) ---
    'ORCHARD': 'CCR', 'SOMERSET': 'CCR', 'RIVER VALLEY': 'CCR',
    'TANGLIN': 'CCR', 'BUKIT TIMAH': 'CCR', 'HOLLAND': 'CCR',
    'NEWTON': 'CCR', 'NOVENA': 'CCR', 'DUNEARN': 'CCR', 'WATTEN': 'CCR',
    'BOAT QUAY': 'CCR', 'RAFFLES PLACE': 'CCR', 'MARINA DOWNTOWN': 'CCR', 'SUNTEC CITY': 'CCR',
    'SHENTON WAY': 'CCR', 'TANJONG PAGAR': 'CCR',
    'SENTOSA': 'CCR',
    'CITY HALL': 'CCR',
    'BUGIS': 'CCR',
    'CENTRAL AREA': 'CCR', # Official HDB Town

    # --- RCR (Rest of Central Region) ---
    'MARINA SOUTH': 'RCR',
    'CHINATOWN': 'RCR',
    'QUEENSTOWN': 'RCR', 'ALEXANDRA': 'RCR', 'TIONG BAHRU': 'RCR',
    'HARBOURFRONT': 'RCR', 'KEPPEL': 'RCR', 'TELOK BLANGAH': 'RCR',
    'BUONA VISTA': 'RCR', 'DOVER': 'RCR', 'PASIR PANJANG': 'RCR',
    'FORT CANNING': 'RCR',
    'ROCHOR': 'RCR',
    'LITTLE INDIA': 'RCR', 'FARRER PARK': 'RCR',
    'BALESTIER': 'RCR', 'WHAMPOA': 'RCR', 'TOA PAYOH': 'RCR', 'BOON KENG': 'RCR', 'BENDEMEER': 'RCR', 'KAMPONG BUGIS': 'RCR',
    'POTONG PASIR': 'RCR', 'BIDADARI': 'RCR', 'MACPHERSON': 'RCR', 'UPPER ALJUNIED': 'RCR',
    'GEYLANG': 'RCR', 'DAKOTA': 'RCR', 'PAYA LEBAR CENTRAL': 'RCR', 'EUNOS': 'RCR', 'UBI': 'RCR', 'ALJUNIED': 'RCR',
    'TANJONG RHU': 'RCR', 'AMBER': 'RCR', 'MEYER': 'RCR', 'KATONG': 'RCR', 'DUNMAN': 'RCR', 'JOO CHIAT': 'RCR', 'MARINE PARADE': 'RCR',
    'BISHAN': 'RCR', 'THOMSON': 'RCR',
    'BUKIT MERAH': 'RCR',      # Official HDB Town
    'KALLANG/WHAMPOA': 'RCR',  # Official HDB Town

    # --- OCR (Outside Central Region) ---
    'CLEMENTI': 'OCR', 'WEST COAST': 'OCR',
    'KEMBANGAN': 'OCR', 'KAKI BUKIT': 'OCR',
    'TELOK KURAU': 'OCR', 'SIGLAP': 'OCR', 'FRANKEL': 'OCR',
    'BEDOK': 'OCR', 'UPPER EAST COAST': 'OCR', 'BAYSHORE': 'OCR', 'TANAH MERAH': 'OCR', 'UPPER CHANGI': 'OCR',
    'FLORA DRIVE': 'OCR', 'LOYANG': 'OCR', 'CHANGI': 'OCR',
    'TAMPINES': 'OCR', 'PASIR RIS': 'OCR',
    'PUNGGOL': 'OCR', 'SENGKANG': 'OCR', 'HOUGANG': 'OCR', 'KOVAN': 'OCR', 'SERANGOON': 'OCR', 'LORONG AH SOO': 'OCR',
    'ANG MO KIO': 'OCR',
    'UPPER BUKIT TIMAH': 'OCR', 'ULU PANDAN': 'OCR', 'CLEMENTI PARK': 'OCR',
    'JURONG EAST': 'OCR', 'JURONG WEST': 'OCR', 'BOON LAY': 'OCR',
    'HILLVIEW': 'OCR', 'BUKIT PANJANG': 'OCR', 'BUKIT BATOK': 'OCR', 'CHOA CHU KANG': 'OCR',
    'KRANJI': 'OCR', 'LIM CHU KANG': 'OCR', 'SUNGEI GEDONG': 'OCR', 'TENGAH': 'OCR',
    'WOODLANDS': 'OCR', 'ADMIRALTY': 'OCR',
    'LENTOR': 'OCR', 'SPRINGLEAF': 'OCR', 'MANDAI': 'OCR',
    'YISHUN': 'OCR', 'SEMBAWANG': 'OCR',
    'SELETAR': 'OCR', 'SELETAR HILL': 'OCR', 'SENGKANG WEST': 'OCR'
}

# Apply the mapping. 
# Using .str.upper() ensures that even if your dataframe has 'Orchard' or 'orchard', it will match the uppercase keys.
final_df['region'] = final_df['hdb_town'].str.upper().map(region_mapping)
final_df['region'] = final_df['region'].fillna('Other')
print("✓ Created region feature")

# Flag mature vs non-mature estates
mature_estates = ['ANG MO KIO', 'BEDOK', 'BISHAN', 'BUKIT MERAH', 'BUKIT TIMAH',
                  'CENTRAL AREA', 'CLEMENTI', 'GEYLANG', 'KALLANG/WHAMPOA',
                  'MARINE PARADE', 'PASIR RIS', 'QUEENSTOWN', 'SERANGOON', 'TAMPINES', 'TOA PAYOH']

final_df['is_mature_estate'] = final_df['hdb_town'].str.upper().isin(mature_estates).astype(int)
print("✓ Created is_mature_estate flag")

✓ Created region feature
✓ Created is_mature_estate flag


In [20]:
final_df['is_mature_estate'].value_counts()

is_mature_estate
0    6718
1    5474
Name: count, dtype: int64

In [21]:
# Remove duplicate rows if any
print(final_df.duplicated().sum())
final_df = final_df.drop_duplicates()

0


In [22]:
final_df.columns.tolist()

['listing_url',
 'exact_address',
 'price_detail',
 'town_detail',
 'bedrooms_detail',
 'bathrooms_detail',
 'area_detail',
 'floor_detail',
 'property_type_detail',
 'address_from_url',
 'listing_id',
 'postal_code',
 'onemap_full_address',
 'onemap_road_name',
 'latitude',
 'longitude',
 'nearest_mrt_name',
 'nearest_mrt_exit',
 'nearest_mrt_distance_m',
 'num_mrt_within_1000m',
 'nearest_clinic_name',
 'nearest_clinic_distance_m',
 'nearest_park_name',
 'nearest_park_distance_m',
 'nearest_community_club_name',
 'nearest_community_club_distance_m',
 'nearest_hawker_name',
 'nearest_hawker_distance_m',
 'num_clinic_within_1000m',
 'num_park_within_1000m',
 'num_community_club_within_1000m',
 'num_hawker_within_1000m',
 'num_amenities_within_1000m',
 'room_count_proxy',
 'floor_area_sqm',
 'hdb_town',
 'region',
 'is_mature_estate']

In [23]:
# See all values in a clean readable format
for value, count in final_df['floor_detail'].value_counts().items():
    print(f"{count:>6} | {value}")

  2122 | High Floor
  1399 | High floor
   727 | high floor
   329 | Mid floor
   324 | Mid Floor
   264 | Low floor
   171 | Low Floor
   122 | low floor
   115 | mid floor
    65 | HIGH FLOOR
    35 | 50th floor
    30 | a high floor
    21 | 2nd floor
    21 | MID FLOOR
    13 | 10th floor
    10 | 2nd Floor
    10 | LOW FLOOR
    10 | 20th floor
     9 | 6th floor
     9 | 4th Floor
     7 | 8th Floor
     6 | a low floor
     6 | 5th floor
     6 | 7th floor
     6 | 3rd floor
     6 | 4th floor
     5 | 3rd Floor
     4 | 15th floor
     4 | 10th Floor
     4 | 8th floor
     4 | 1st floor
     4 | 12th Floor
     4 | 6th Floor
     4 | 11th floor
     4 | a mid floor
     4 | High mid-floor position for privacy and elevated views
     3 | high level
     3 | high Floor
     3 | HIGH floor
     3 | 30th floor
     3 | a Super high floor
     3 | HDB flat type
     2 | 12th floor
     2 | 11th Floor
     2 | a high floor and North-South facing
     2 | 14th Floor
     2 | 11TH FLO

In [24]:
import re
import numpy as np
import pandas as pd

# ============================================================
# STANDARDISED FLOOR RANGES (consistent throughout)
# Low:  Floor 1  - 5
# Mid:  Floor 6  - 12
# High: Floor 13+
# ============================================================

def map_floor_category(text):
    if pd.isna(text) or str(text).strip() == '':
        return np.nan

    text_clean = str(text).lower().strip()

    # ============================================================
    # RULE 1: Extract explicit floor numbers
    # Handles: "10th floor", "2nd floor", "Level 11", "09 floor"
    # ============================================================
    # Handle "level X" format
    level_match = re.search(r'level\s*(\d+)', text_clean)
    if level_match:
        floor_num = int(level_match.group(1))
        return num_to_category(floor_num)

    # Handle "Xth/Xnd/Xrd/Xst floor" format
    number_match = re.search(r'\b(\d+)\s*(?:st|nd|rd|th)?\s*floor', text_clean)
    if number_match:
        floor_num = int(number_match.group(1))
        return num_to_category(floor_num)

    # Handle "floor X" format (e.g. "floor 9")
    floor_num_match = re.search(r'floor\s*(\d+)', text_clean)
    if floor_num_match:
        floor_num = int(floor_num_match.group(1))
        return num_to_category(floor_num)

    # ============================================================
    # RULE 2: Handle "above X" phrases
    # Handles: "above level 6", "above 10", "12 and above"
    # ============================================================
    above_match = re.search(r'above\s*(?:level|the)?\s*(\d+)', text_clean)
    if above_match:
        floor_num = int(above_match.group(1))
        return num_to_category(floor_num + 1)  # "above 6" = floor 7

    and_above_match = re.search(r'(\d+)\s*and\s*above', text_clean)
    if and_above_match:
        floor_num = int(and_above_match.group(1))
        return num_to_category(floor_num)

    # ============================================================
    # RULE 3: Standalone numbers only
    # Handles: "15", "32", "4", "17"
    # Must come BEFORE keyword rules to avoid matching
    # numbers inside garbage text like "3 bedrooms"
    # ============================================================
    standalone_match = re.fullmatch(r'\s*(\d+)\s*', text_clean)
    if standalone_match:
        floor_num = int(standalone_match.group(1))
        return num_to_category(floor_num)

    # ============================================================
    # RULE 4: Keyword combinations
    # Order matters: most specific phrases checked first
    # ============================================================

    # Super / very high → high
    if any(phrase in text_clean for phrase in [
        'super high', 'very high', 'highest', 'top floor', 'top level',
        'a very high', 'a super high'
    ]):
        return 'high'

    # Mid to high / mid-high → high (lean towards high)
    if any(phrase in text_clean for phrase in [
        'mid to high', 'mid-to-high', 'mid-high', 'mid high'
    ]):
        return 'high'

    # High (all variations)
    if any(phrase in text_clean for phrase in [
        'high floor', 'high level', 'higher floor',
        'high unit', 'high uni', ' high '
    ]):
        return 'high'
    if text_clean in ['high']:
        return 'high'

    # Mid (all variations)
    if any(phrase in text_clean for phrase in [
        'mid floor', 'mid level', 'mid-floor', 'mid-level',
        'middle floor', 'mid unit', 'mid '
    ]):
        return 'mid'
    if text_clean in ['mid']:
        return 'mid'

    # Low (all variations)
    if any(phrase in text_clean for phrase in [
        'low floor', 'low level', 'lower floor', 'low unit',
        'ground floor', '1st floor', 'corridor level', 'low '
    ]):
        return 'low'
    if text_clean in ['low']:
        return 'low'

    # ============================================================
    # RULE 5: Catch remaining directional keywords
    # Handles: "a higher floor", "the lower floor"
    # ============================================================
    if 'lower' in text_clean:
        return 'low'
    if 'higher' in text_clean:
        return 'high'

    # ============================================================
    # RULE 6: Truly unclassifiable garbage values
    # Handles: "Bendemeer Road", "3 bedrooms", "Canberra Street"
    # Return NaN and let the fallback fill them in
    # ============================================================
    return np.nan


def num_to_category(floor_num):
    """
    Standardised floor number to category mapping.
    Consistent ranges used throughout the entire function.
    Low:  1  - 5
    Mid:  6  - 12
    High: 13+
    """
    if floor_num <= 5:
        return 'low'
    elif floor_num <= 12:
        return 'mid'
    else:
        return 'high'


# ============================================================
# APPLY AND HANDLE ALL NaN VALUES
# ============================================================

# Step 1: Apply the mapping function
final_df['floor_category'] = final_df['floor_detail'].apply(map_floor_category).fillna('mid')

print(final_df['floor_category'].value_counts(dropna=False))
print(f"NaN count: {final_df['floor_category'].isna().sum()}")

# Step 2: Map floor category to numeric storey_mid
floor_to_storey = {
    'low':  3,   # Midpoint of floors 1-5
    'mid':  9,   # Midpoint of floors 6-12
    'high': 16   # Representative of floors 13+
}
final_df['storey_mid'] = final_df['floor_category'].map(floor_to_storey)

floor_category
mid     7040
high    4484
low      668
Name: count, dtype: int64
NaN count: 0


In [25]:
def infer_flat_type(floor_area_sqm, room_count=None):
    """
    Infers HDB flat type from floor area and optional room count.
    
    Parameters
    ----------
    floor_area_sqm : float
    room_count     : int or None
    
    Returns
    -------
    str : flat type label e.g. '4 ROOM'
    """
    if pd.isna(floor_area_sqm):
        return np.nan

    if floor_area_sqm < 52:
        return '2 ROOM'
    elif floor_area_sqm <= 82:
        # room_count disambiguates — if 3 rooms, confirm as 3 ROOM
        # otherwise could be a small 4 ROOM
        if room_count is not None and not pd.isna(room_count):
            return '3 ROOM' if int(room_count) == 3 else '4 ROOM'
        return '3 ROOM'   # default assumption in this range
    elif floor_area_sqm <= 105:
        return '4 ROOM'
    else:
        return '5 ROOM'


# ============================================================
# Apply to your dataframe
# ============================================================
final_df['flat_type'] = final_df.apply(
    lambda row: infer_flat_type(row['floor_area_sqm'], row.get('room_count')),
    axis=1
)

In [26]:
final_df['flat_type'].value_counts()

flat_type
4 ROOM    5432
5 ROOM    3970
3 ROOM    2540
2 ROOM     250
Name: count, dtype: int64

In [27]:
import pandas as pd
import re

def extract_block(text: str) -> str:
    match = re.search(r"^\D*(\d+[A-Za-z]?)", text.strip()) if pd.notna(text) else None
    return match.group(1) if match else ""

final_df["block_number"] = final_df["onemap_full_address"].apply(extract_block)
final_df["standardised_address"] = final_df["block_number"] + " " + final_df["onemap_road_name"].fillna("")

In [28]:
final_df['standardised_address'].value_counts().head(25)

standardised_address
104A BIDADARI PARK DRIVE          39
118A ALKAFF CRESCENT              39
1 CANTONMENT ROAD                 38
94 DAWSON ROAD                    28
467B BUKIT BATOK WEST AVENUE 9    23
103B BIDADARI PARK DRIVE          20
422A NORTHSHORE DRIVE             20
88 DAWSON ROAD                    19
101A BIDADARI PARK DRIVE          19
228A ANG MO KIO STREET 23         19
96 DAWSON ROAD                    18
464A BUKIT BATOK WEST AVENUE 8    18
409D NORTHSHORE DRIVE             17
228 TAMPINES STREET 23            17
227A ANG MO KIO STREET 23         16
86 DAWSON ROAD                    16
95 DAWSON ROAD                    16
103A BIDADARI PARK DRIVE          15
453B BUKIT BATOK WEST AVENUE 6    15
93B TELOK BLANGAH STREET 31       15
90A TELOK BLANGAH STREET 31       15
452A BUKIT BATOK WEST AVENUE 6    15
619A TAMPINES STREET 61           14
618A TAMPINES STREET 61           14
775 YISHUN RING ROAD              14
Name: count, dtype: int64

In [30]:
# ─────────────────────────────────────────────────────────────────
# 2. NORMALISE
#    Fixes encoding, strips "Blk/Block", expands abbreviations,
#    title-cases, and collapses whitespace.
# ─────────────────────────────────────────────────────────────────
_ABBREV_MAP = [
    # Road types
    (r"\bAve\b",    "Avenue"),
    (r"\bAv\b",     "Avenue"),
    (r"\bSt\b",     "Street"),
    (r"\bRd\b",     "Road"),
    (r"\bDr\b",     "Drive"),
    (r"\bCres\b",   "Crescent"),
    (r"\bCrst\b",   "Crescent"),
    (r"\bCrt\b",    "Crescent"),
    (r"\bCl\b",     "Close"),
    (r"\bCls\b",    "Close"),
    (r"\bLn\b",     "Lane"),
    (r"\bTce\b",    "Terrace"),
    (r"\bTer\b",    "Terrace"),
    (r"\bBvd\b",    "Boulevard"),
    (r"\bBlvd\b",   "Boulevard"),
    (r"\bWk\b",     "Walk"),
    (r"\bWy\b",     "Way"),
    (r"\bLk\b",     "Link"),
    (r"\bHwy\b",    "Highway"),
    (r"\bRise\b",   "Rise"),       # already full but kept for consistency

    # Directional
    (r"\bNth\b",    "North"),
    (r"\bNth\.\b",  "North"),
    (r"\bSth\b",    "South"),
    (r"\bSth\.\b",  "South"),
    (r"\bEst\b",    "East"),
    (r"\bWst\b",    "West"),
    (r"\bNE\b",     "North East"),
    (r"\bNW\b",     "North West"),
    (r"\bSE\b",     "South East"),
    (r"\bSW\b",     "South West"),

    # Common words
    (r"\bCtrl\b",   "Central"),
    (r"\bCtr\b",    "Central"),
    (r"\bCent\b",   "Central"),
    (r"\bCentral\b","Central"),    # already full
    (r"\bJln\b",    "Jalan"),
    (r"\bJl\b",     "Jalan"),
    (r"\bLor\b",    "Lorong"),
    (r"\bLrg\b",    "Lorong"),
    (r"\bHts\b",    "Heights"),
    (r"\bHt\b",     "Heights"),
    (r"\bPl\b",     "Place"),
    (r"\bGdn\b",    "Garden"),
    (r"\bGdns\b",   "Gardens"),
    (r"\bVw\b",     "View"),
    (r"\bPk\b",     "Park"),
    (r"\bPrk\b",    "Park"),
    (r"\bGrn\b",    "Green"),
    (r"\bGr\b",     "Green"),
    (r"\bGlen\b",   "Glen"),       # already full
    (r"\bRes\b",    "Reservoir"),
    (r"\bRsvr\b",   "Reservoir"),
    (r"\bMkt\b",    "Market"),
    (r"\bStn\b",    "Station"),
    (r"\bHse\b",    "House"),
    (r"\bBldg\b",   "Building"),
    (r"\bApt\b",    "Apartment"),
    (r"\bApts\b",   "Apartments"),

    # Singapore-specific area prefixes/words
    (r"\bBt\b",     "Bukit"),
    (r"\bBkt\b",    "Bukit"),
    (r"\bBk\b",     "Bukit"),
    (r"\bTg\b",     "Tanjong"),
    (r"\bTjg\b",    "Tanjong"),
    (r"\bTanj\b",   "Tanjong"),
    (r"\bKg\b",     "Kampong"),
    (r"\bKpg\b",    "Kampong"),
    (r"\bKamp\b",   "Kampong"),
    (r"\bPsr\b",    "Pasir"),
    (r"\bPg\b",     "Punggol"),
    (r"\bAMK\b",    "Ang Mo Kio"),
    (r"\bBBK\b",    "Bukit Batok"),
    (r"\bBP\b",     "Bukit Panjang"),
    (r"\bBM\b",     "Bukit Merah"),
    (r"\bBT\b",     "Bukit Timah"),
    (r"\bCCK\b",    "Choa Chu Kang"),
    (r"\bHG\b",     "Hougang"),
    (r"\bJE\b",     "Jurong East"),
    (r"\bJW\b",     "Jurong West"),
    (r"\bTB\b",     "Telok Blangah"),
    (r"\bWL\b",     "Woodlands"),
    (r"\bYS\b",     "Yishun"),
    (r"\bSKG\b",    "Sengkang"),
    (r"\bSKW\b",    "Sengkang West"),
    (r"\bTPY\b",    "Toa Payoh"),
    (r"\bSRN\b",    "Serangoon"),
    (r"\bTPS\b",    "Tampines"),
    (r"\bPR\b",     "Pasir Ris"),
    (r"\bBDK\b",    "Bedok"),
    (r"\bCLM\b",    "Clementi"),
]

def normalise(addr: str) -> str:
    s = addr.strip()
    if not s:
        return ""

    # Fix mojibake (Â + non-breaking space from latin1/UTF-8 mismatch)
    s = s.replace("Â\xa0", " ").replace("Â ", " ").replace("\xa0", " ").replace("Â", "")

    # Strip leading Block / Blk (with or without space before number, e.g. "Blk301")
    s = re.sub(r"^(Block|Blk)\s*", "", s, flags=re.IGNORECASE)

    # Remove dots used as separators between letter and digit (e.g. "St.31" → "St 31")
    s = re.sub(r"(?<=[A-Za-z])\.(?=\d)", " ", s)

    # Strip trailing marketing noise after "!" (keep the address part)
    s = re.sub(r"\s*!\s.*$", "", s).strip()

    # Expand abbreviations
    for pattern, replacement in _ABBREV_MAP:
        s = re.sub(pattern, replacement, s, flags=re.IGNORECASE)

    # Title-case
    s = s.title()

    # Restore lowercase for common prepositions broken by title-case
    for word in ("the", "of", "at", "to", "and", "a"):
        s = re.sub(r"\b" + word.title() + r"\b", word, s)

    # Collapse whitespace
    return re.sub(r"\s{2,}", " ", s).strip()

In [31]:
# Apply normalise to both columns
final_df["address_norm"] = final_df["standardised_address"].apply(lambda x: normalise(str(x)) if pd.notna(x) else "")

# Download from: https://data.gov.sg/dataset/hdb-property-information
hdb_info_df = pd.read_csv('HDBPropertyInformation.csv')

# Create a matching address key in the HDB dataset
# Combines block + street e.g. "123 ANG MO KIO AVE 3"
hdb_info_df['address'] = (
    hdb_info_df['blk_no'].astype(str).str.strip() + " " +
    hdb_info_df['street'].astype(str).str.strip()
)
hdb_info_df['address'] = hdb_info_df['address'].str.upper().str.strip()

hdb_info_df["address_norm"] = hdb_info_df["address"].apply(lambda x: normalise(str(x)) if pd.notna(x) else "")

In [32]:
final_df['address_norm'].value_counts().sample(10)

address_norm
156 Lorong 1 Toa Payoh           3
519D Tampines Central 8          1
309 Serangoon Avenue 2           5
287 Bukit Batok East Avenue 3    1
319 Tampines Street 33           1
304 Ang Mo Kio Avenue 1          2
626 Senja Road                   1
121 Mcnair Road                  7
650B Jurong West Street 61       3
464A Clementi Avenue 1           1
Name: count, dtype: int64

In [33]:
hdb_info_df['address_norm'].value_counts().sample(10)

address_norm
171 Bedok South Road       1
324 Tengah Garden Walk     1
450G Tampines Street 42    1
980D Buangkok Crescent     1
273 Tampines Street 22     1
209 New Upp Changi Road    1
237B Boon Lay Drive        1
741 Yishun Avenue 5        1
283 Bishan Street 22       1
307A Anchorvale Road       1
Name: count, dtype: int64

In [34]:
# ============================================================
# LEASE FEATURES: Merge with HDB Property Information
# ============================================================

# Merge the datasets on address_norm
# We only need 'year_completed' from the HDB info dataset
final_df = pd.merge(
    final_df,
    hdb_info_df[['address_norm', 'year_completed', 'max_floor_lvl']].drop_duplicates('address_norm'),
    on='address_norm',
    how='left'
)

# Calculate lease features
CURRENT_YEAR = datetime.now().year

final_df['lease_commence_date'] = final_df['year_completed']
final_df['flat_age_at_sale'] = CURRENT_YEAR - final_df['lease_commence_date']
final_df['remaining_lease'] = 99 - final_df['flat_age_at_sale']
final_df['sold_year'] = CURRENT_YEAR

# Check how many addresses matched successfully
matched = final_df['lease_commence_date'].notna().sum()
total = len(final_df)
print(f"Successfully matched: {matched}/{total} listings ({matched/total*100:.1f}%)")

# Fill any unmatched addresses with town-level median (fallback)
for col in ['lease_commence_date', 'flat_age_at_sale', 'remaining_lease']:
    final_df[col] = final_df.groupby('hdb_town')[col].transform(
        lambda x: x.fillna(x.median())
    )

# Drop temporary columns
#final_df = final_df.drop(columns=['address_key', 'year_completed'])

print("Lease features done.")
print(final_df[['onemap_full_address', 'lease_commence_date', 'remaining_lease', 'flat_age_at_sale']].head(10))

Successfully matched: 11951/12192 listings (98.0%)
Lease features done.
                                 onemap_full_address  lease_commence_date  \
0  903 TAMPINES AVENUE 4 TAMPINES PALMSVILLE SING...               1983.0   
1           32 BEDOK SOUTH AVENUE 2 SINGAPORE 460032               1977.0   
2         224 JURONG EAST STREET 21 SINGAPORE 600224               1984.0   
3  302 CLEMENTI AVENUE 4 CLEMENTI MEADOWS SINGAPO...               1978.0   
4   94 DAWSON ROAD SKYPARC @ DAWSON SINGAPORE 141094               2020.0   
5                   181 JELEBU ROAD SINGAPORE 670181               2001.0   
6               18 BEDOK SOUTH ROAD SINGAPORE 460018               1975.0   
7         226 JURONG EAST STREET 21 SINGAPORE 600226               1982.0   
8  706 YISHUN AVENUE 5 CHONG PANG GREEN SINGAPORE...               1983.0   
9  116A JALAN TENTERAM TENTERAM PEAK SINGAPORE 32...               2016.0   

   remaining_lease  flat_age_at_sale  
0             56.0              43.0  
1 

In [35]:
final_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12192 entries, 0 to 12191
Data columns (total 50 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   listing_url                        12192 non-null  str    
 1   exact_address                      12192 non-null  str    
 2   price_detail                       12192 non-null  str    
 3   town_detail                        12192 non-null  str    
 4   bedrooms_detail                    12192 non-null  float64
 5   bathrooms_detail                   11823 non-null  float64
 6   area_detail                        12192 non-null  str    
 7   floor_detail                       6082 non-null   str    
 8   property_type_detail               12192 non-null  str    
 9   address_from_url                   12192 non-null  str    
 10  listing_id                         12192 non-null  int64  
 11  postal_code                        12185 non-null  float64
 12  o

In [36]:
final_df.sample(10)

,listing_url,exact_address,price_detail,town_detail,bedrooms_detail,bathrooms_detail,area_detail,floor_detail,property_type_detail,address_from_url,...,flat_type,block_number,standardised_address,address_norm,year_completed,max_floor_lvl,lease_commence_date,flat_age_at_sale,remaining_lease,sold_year
10760,https://www.propertyguru.com.sg/listing/hdb-fo...,936 Jurong West Street 91,"S$ 428,000",Clementi,2.0,2.0,763 sqft,NaN,HDB,936 Jurong West Street 91,...,3 ROOM,936,936 JURONG WEST STREET 91,936 Jurong West Street 91,1986.0,12.0,1986.0,40.0,59.0,2026
2907,https://www.propertyguru.com.sg/listing/hdb-fo...,101 Whampoa Drive,"S$ 550,000",Clementi,3.0,2.0,980 sqft,High Floor,HDB,101 Whampoa Drive,...,4 ROOM,101,101 WHAMPOA DRIVE,101 Whampoa Drive,1973.0,25.0,1973.0,53.0,46.0,2026
12042,https://www.propertyguru.com.sg/listing/hdb-fo...,672C Yishun Avenue 4,"S$ 580,000",Clementi,3.0,2.0,"1,001 sqft",NaN,HDB,672C Yishun Avenue 4,...,4 ROOM,672C,672C YISHUN AVENUE 4,672C Yishun Avenue 4,2016.0,13.0,2016.0,10.0,89.0,2026
5386,https://www.propertyguru.com.sg/listing/hdb-fo...,65 Kallang Bahru,"S$ 468,000",Clementi,3.0,1.0,700 sqft,Mid floor,HDB,65 Kallang Bahru,...,3 ROOM,65,65 KALLANG BAHRU,65 Kallang Bahru,1973.0,13.0,1973.0,53.0,46.0,2026
9685,https://www.propertyguru.com.sg/listing/hdb-fo...,953 Hougang Avenue 9,"S$ 650,000",Clementi,3.0,2.0,"1,087 sqft",NaN,HDB,953 Hougang Avenue 9,...,4 ROOM,953,953 HOUGANG AVENUE 9,953 Hougang Avenue 9,1997.0,16.0,1997.0,29.0,70.0,2026
11959,https://www.propertyguru.com.sg/listing/hdb-fo...,135 Lorong Ah Soo,"S$ 730,000",Clementi,3.0,2.0,"1,421 sqft",Low floor,HDB,135 Lorong Ah Soo,...,5 ROOM,135,135 LORONG AH SOO,135 Lorong Ah Soo,1984.0,15.0,1984.0,42.0,57.0,2026
2103,https://www.propertyguru.com.sg/listing/hdb-fo...,61C Strathmore Avenue,"S$ 1,150,000",Clementi,3.0,2.0,"1,012 sqft",High Floor,HDB,61C Strathmore Avenue,...,4 ROOM,61C,61C STRATHMORE AVENUE,61C Strathmore Avenue,2010.0,30.0,2010.0,16.0,83.0,2026
709,https://www.propertyguru.com.sg/listing/hdb-fo...,50 Dorset Road,"S$ 510,000",Clementi,2.0,2.0,743 sqft,NaN,HDB,50 Dorset Road,...,3 ROOM,50,50 DORSET ROAD,50 Dorset Road,1978.0,12.0,1978.0,48.0,51.0,2026
9522,https://www.propertyguru.com.sg/listing/hdb-fo...,460 Ang Mo Kio Avenue 10,"S$ 795,000",Ang Mo Kio,3.0,2.0,"1,292 sqft",high floor,HDB,460 Ang Mo Kio Avenue 10,...,5 ROOM,460,460 ANG MO KIO AVENUE 10,460 Ang Mo Kio Avenue 10,1979.0,25.0,1979.0,47.0,52.0,2026
3437,https://www.propertyguru.com.sg/listing/hdb-fo...,204A Compassvale Drive,"S$ 599,000",Clementi,3.0,2.0,990 sqft,NaN,HDB,204A Compassvale Drive,...,4 ROOM,204A,204A COMPASSVALE DRIVE,204A Compassvale Drive,1999.0,16.0,1999.0,27.0,72.0,2026


In [37]:
final_df.columns.tolist()

['listing_url',
 'exact_address',
 'price_detail',
 'town_detail',
 'bedrooms_detail',
 'bathrooms_detail',
 'area_detail',
 'floor_detail',
 'property_type_detail',
 'address_from_url',
 'listing_id',
 'postal_code',
 'onemap_full_address',
 'onemap_road_name',
 'latitude',
 'longitude',
 'nearest_mrt_name',
 'nearest_mrt_exit',
 'nearest_mrt_distance_m',
 'num_mrt_within_1000m',
 'nearest_clinic_name',
 'nearest_clinic_distance_m',
 'nearest_park_name',
 'nearest_park_distance_m',
 'nearest_community_club_name',
 'nearest_community_club_distance_m',
 'nearest_hawker_name',
 'nearest_hawker_distance_m',
 'num_clinic_within_1000m',
 'num_park_within_1000m',
 'num_community_club_within_1000m',
 'num_hawker_within_1000m',
 'num_amenities_within_1000m',
 'room_count_proxy',
 'floor_area_sqm',
 'hdb_town',
 'region',
 'is_mature_estate',
 'floor_category',
 'storey_mid',
 'flat_type',
 'block_number',
 'standardised_address',
 'address_norm',
 'year_completed',
 'max_floor_lvl',
 'lease_co

In [38]:
# ============================================================
# Drop Unnecessary Columns
# ============================================================
# Keep only columns needed for prediction, SAI scoring,
# and final output display
cols_to_keep = [
    # Identifiers and display
    'listing_url', 'onemap_full_address', 'postal_code',
    'price_detail', 'hdb_town', 

    # Model features
    'floor_area_sqm', 'flat_type', 'lease_commence_date',
    'remaining_lease', 'flat_age_at_sale', 'sold_year',
    'storey_mid', 'floor_category', 'region', 'is_mature_estate',
    'nearest_mrt_distance_m', 'nearest_clinic_distance_m',
    'nearest_park_distance_m', 'nearest_community_club_distance_m',
    'nearest_hawker_distance_m', 'num_mrt_within_1000m',
    'num_clinic_within_1000m', 'num_park_within_1000m',
    'num_community_club_within_1000m', 'num_hawker_within_1000m',
    'num_amenities_within_1000m', 'max_floor_lvl'

    # SAI display columns
    'nearest_mrt_name', 'nearest_community_club_name',
]

# Only keep columns that actually exist in the dataframe
cols_to_keep = [c for c in cols_to_keep if c in final_df.columns]
final_df = final_df[cols_to_keep].copy()

print(f"Final shape: {final_df.shape}")
print(f"Columns kept: {final_df.columns.tolist()}")


Final shape: (12192, 27)
Columns kept: ['listing_url', 'onemap_full_address', 'postal_code', 'price_detail', 'hdb_town', 'floor_area_sqm', 'flat_type', 'lease_commence_date', 'remaining_lease', 'flat_age_at_sale', 'sold_year', 'storey_mid', 'floor_category', 'region', 'is_mature_estate', 'nearest_mrt_distance_m', 'nearest_clinic_distance_m', 'nearest_park_distance_m', 'nearest_community_club_distance_m', 'nearest_hawker_distance_m', 'num_mrt_within_1000m', 'num_clinic_within_1000m', 'num_park_within_1000m', 'num_community_club_within_1000m', 'num_hawker_within_1000m', 'num_amenities_within_1000m', 'nearest_community_club_name']


In [39]:
final_df["num_hawker_within_1000m"] = final_df["num_hawker_within_1000m"].replace(0, 1) # Base value of 1

In [41]:
final_df.sample(10)

,listing_url,onemap_full_address,postal_code,price_detail,hdb_town,floor_area_sqm,flat_type,lease_commence_date,remaining_lease,flat_age_at_sale,...,nearest_park_distance_m,nearest_community_club_distance_m,nearest_hawker_distance_m,num_mrt_within_1000m,num_clinic_within_1000m,num_park_within_1000m,num_community_club_within_1000m,num_hawker_within_1000m,num_amenities_within_1000m,nearest_community_club_name
10783,https://www.propertyguru.com.sg/listing/hdb-fo...,458 YISHUN AVENUE 11 DEW SPRING @ YISHUN SINGA...,760458.0,"S$ 583,888",Yishun,92.995903,4 ROOM,2012.0,85.0,14.0,...,343.5,483.5,2299.5,0.0,15.0,2.0,1.0,1.0,18.0,Yishun Link CC
10751,https://www.propertyguru.com.sg/listing/hdb-fo...,112 TECK WHYE LANE SINGAPORE 680112,680112.0,"S$ 545,000",Choa Chu Kang,106.002323,5 ROOM,1988.0,61.0,38.0,...,501.9,405.8,4077.2,5.0,14.0,5.0,1.0,1.0,20.0,Chua Chu Kang CC
2137,https://www.propertyguru.com.sg/listing/hdb-fo...,368 TAMPINES STREET 34 SINGAPORE 520368,520368.0,"S$ 668,000",Tampines,102.007494,4 ROOM,1995.0,68.0,31.0,...,536.9,933.2,2352.8,1.0,14.0,3.0,1.0,1.0,18.0,Tampines East CC
5405,https://www.propertyguru.com.sg/listing/hdb-fo...,86 DAWSON ROAD SKYVILLE @ DAWSON SINGAPORE 141086,141086.0,"S$ 1,399,999",Queenstown,103.958457,4 ROOM,2015.0,88.0,11.0,...,1126.0,800.3,857.9,1.0,15.0,0.0,1.0,2.0,18.0,Leng Kee CC
10277,https://www.propertyguru.com.sg/listing/hdb-fo...,88 DAWSON ROAD SKYVILLE @ DAWSON SINGAPORE 142088,142088.0,"S$ 980,000",Queenstown,82.962379,4 ROOM,2015.0,88.0,11.0,...,1152.6,879.8,762.3,1.0,13.0,0.0,2.0,2.0,17.0,Leng Kee CC
798,https://www.propertyguru.com.sg/listing/hdb-fo...,126A KIM TIAN ROAD KIM TIAN GREEN SINGAPORE 16...,161126.0,"S$ 1,400,000",Bukit Merah,112.970048,5 ROOM,2011.0,84.0,15.0,...,47.0,273.6,367.2,3.0,22.0,6.0,3.0,5.0,36.0,Tiong Bahru CC
5440,https://www.propertyguru.com.sg/listing/hdb-fo...,290C BUKIT BATOK EAST AVENUE 3 SPRING VIEW SIN...,650290.0,"S$ 950,000",Bukit Batok,141.026754,5 ROOM,1996.0,69.0,30.0,...,511.8,550.5,2175.3,0.0,13.0,3.0,1.0,1.0,17.0,Bukit Batok East CC (U/C)
10132,https://www.propertyguru.com.sg/listing/hdb-fo...,751 WOODLANDS CIRCLE SINGAPORE 730751,730751.0,"S$ 680,000",Woodlands,121.981639,5 ROOM,1996.0,69.0,30.0,...,1055.2,889.8,1922.2,1.0,17.0,0.0,1.0,1.0,18.0,Woodlands CC (Pending U/C)
8651,https://www.propertyguru.com.sg/listing/hdb-fo...,611 ANG MO KIO AVENUE 5 PCF SPARKLETOTS PRESCH...,560611.0,"S$ 400,000",Ang Mo Kio,66.983063,3 ROOM,1979.0,52.0,47.0,...,212.0,585.7,468.1,3.0,17.0,15.0,2.0,2.0,36.0,Kebun Baru CC
1996,https://www.propertyguru.com.sg/listing/hdb-fo...,20 JALAN KLINIK SINGAPORE 160020,160020.0,"S$ 460,000",Bukit Merah,68.004996,3 ROOM,1963.0,36.0,63.0,...,384.1,253.9,66.8,3.0,23.0,4.0,3.0,6.0,36.0,Kim Seng CC


In [50]:
# Amenity saturation thresholds (max counts)

# These values cap the maximum number of amenities counted towards the SAI score.
# This serves two main purposes:
# 1. Diminishing Returns: Having 3 MRTs nearby is excellent, but a 4th doesn't add much practical value.
# 2. Score Normalization: It prevents a location with an extreme abundance of one amenity 
#    (e.g., 20 clinics in a commercial hub) from disproportionately skewing the final SAI score.

max_counts = {
    'clinic': 10,  # cap to avoid over-rewarding dense commercial clusters
    'hawker': 3,   # a few nearby options are enough; extra ones add little value
    'park': 3,     # parks are sparse, so small caps are usually sufficient
    'mrt': 3       # MRT access saturates quickly after 1-3 nearby stations
}

In [51]:
import math
import pandas as pd

def calculate_sai_for_row(row, sliders, half_life_distance=500):
    """
    Calculates the SAI using a lenient Exponential Decay for distance for a single DataFrame row.
    Slider weights for 'clinic', 'hawker', 'park', 'mrt' should be between 1 and 10.
    The final SAI score is guaranteed to be between 0 and 100.
    """
    decay_rate = math.log(2) / half_life_distance
    
    # Extract distances from the row (default to 500m if missing or NaN)
    distances = {
        'clinic': row.get('nearest_clinic_distance_m', 500),
        'hawker': row.get('nearest_hawker_distance_m', 500),
        'park': row.get('nearest_park_distance_m', 500),
        'mrt': row.get('nearest_mrt_distance_m', 500)
    }
    
    # Extract counts from the row (default to 1 if missing or NaN)
    counts = {
        'clinic': row.get('num_clinic_within_1000m', 1),
        'hawker': row.get('num_hawker_within_1000m', 1),
        'park': row.get('num_park_within_1000m', 1),
        'mrt': row.get('num_mrt_within_1000m', 1) 
    }
    
    weighted_sum = 0
    total_weights = 0
    
    for category in ['clinic', 'hawker', 'park', 'mrt']:
        max_c = max_counts.get(category, 1)
        
        # Handle potential NaN values safely and ensure no negative values
        dist = distances[category] if pd.notna(distances[category]) else 500
        dist = max(0, dist) # Prevent negative distances from inflating score > 50
        
        count = counts[category] if pd.notna(counts[category]) else 1
        count = max(0, count) # Prevent negative counts
        
        # Get the slider weight (expecting 1-10, default to 1 if missing)
        weight = sliders.get(category, 1)
        
        # Cap count to max_c to ensure count_score doesn't exceed 50
        count_capped = min(count, max_c)
        
        # 1. Lenient Exponential Decay for Distance (Max 50 points)
        dist_score = 50 * math.exp(-decay_rate * dist)
        
        # 2. Linear scale for Density (Max 50 points)
        count_score = 50 * (count_capped / max_c) if max_c > 0 else 0
        
        # Total Category Score (Max 100 points)
        score = dist_score + count_score
        
        # Applying the individual slider weight to the category score
        weighted_sum += (score * weight)
        # Keeping track of the total slider values used
        total_weights += weight
        
    if total_weights == 0:
        return 0.0

    # Normalization of SAI score     
    final_sai = weighted_sum / total_weights
    
    # Strictly bound the final score between 0 and 100
    final_sai = max(0.0, min(100.0, final_sai))
    
    return round(final_sai, 1)

In [52]:
# ==========================================
# DATAFRAME APPLICATION EXAMPLE
# ==========================================

# 2. Define user sliders mapped to the new categories
# user_sliders = {
#     'clinic': 7, 
#     'hawker': 8, 
#     'park': 4, 
#     'mrt': 9
# }

import re 

def final_output(df_input, town, min_rooms, user_sliders, budget=None):
    """
    Filters the dataframe based on town, minimum room count, and an optional budget.
    """
    df = df_input.copy()

    # ============================================================
    # CLEAN postal_code
    # ============================================================
    df['postal_code'] = df['postal_code'].astype('Int64').astype(str)
    df['postal_code'] = df['postal_code'].replace('<NA>', '')

    # ============================================================
    # CLEAN price_detail ONCE — used for both filtering and display
    # ============================================================
    df['price_clean'] = (
        df['price_detail']
        .replace(r'\D', '', regex=True)
        .fillna('0')
        .astype(int)
    )
    df['formatted_price'] = df['price_clean'].map('${:,.0f}'.format)

    # ============================================================
    # DERIVE room_num from flat_type for filtering
    # ============================================================
    df['room_num'] = df['flat_type'].str.extract(r'(\d+)').astype(float)

    # ============================================================
    # FILTER 1: by town
    # ============================================================
    filtered_df = df[df['hdb_town'].str.lower() == town.lower()].copy()

    # ============================================================
    # FILTER 2: by minimum rooms
    # ============================================================
    filtered_df = filtered_df[filtered_df['room_num'] >= min_rooms]

    # ============================================================
    # FILTER 3: by budget (optional)
    # ============================================================
    if budget is not None:
        filtered_df = filtered_df[filtered_df['price_clean'] <= budget]

    # ============================================================
    # DEDUP: keep lowest price per address
    # ============================================================
    filtered_df = filtered_df.sort_values(
        by=['onemap_full_address', 'price_clean'],
        ascending=[True, True]
    )
    filtered_df = filtered_df.drop_duplicates(
        subset=['onemap_full_address'],
        keep='first'
    )

    # ============================================================
    # SCORE + RANK
    # ============================================================
    filtered_df['SAI_Score'] = filtered_df.apply(
        lambda row: calculate_sai_for_row(row, user_sliders), axis=1
    )

    ranked_df = filtered_df.sort_values(
        by=['SAI_Score', 'price_clean'],
        ascending=[False, True]
    )

    top_3_df = ranked_df.head(3)

    # ============================================================
    # RETURN selected columns
    # ============================================================
    columns_to_return = [
        'listing_url', 'onemap_full_address', 'hdb_town', 'postal_code',
        'formatted_price', 'SAI_Score', 'flat_type', 
        'nearest_mrt_distance_m', 'num_mrt_within_1000m',
        'nearest_clinic_distance_m', 'num_clinic_within_1000m',
        'nearest_community_club_name', 'nearest_community_club_distance_m',
        'nearest_hawker_distance_m', 'num_hawker_within_1000m',
        'nearest_park_distance_m','num_park_within_1000m',                                 
    ]

    return top_3_df[columns_to_return]

In [54]:
# Test Case 1: Filter by town and min_rooms only (no budget)
user_sliders = {
    'clinic': 8, 
    'hawker': 5, 
    'park': 7, 
    'mrt': 10, 
}

print("Test Case 1: Tampines, Min 3 rooms, No budget limit")
result_1 = final_output(final_df, town='Tampines', min_rooms=3, user_sliders=user_sliders)
result_1.T

Test Case 1: Tampines, Min 3 rooms, No budget limit


,9578,5309,327
listing_url,https://www.propertyguru.com.sg/listing/hdb-fo...,https://www.propertyguru.com.sg/listing/hdb-fo...,https://www.propertyguru.com.sg/listing/hdb-fo...
onemap_full_address,228 SIMEI STREET 4 SINGAPORE 520228,505 TAMPINES CENTRAL 1 TAMPINES HEART SINGAPOR...,168C SIMEI LANE PARC LUMIERE SINGAPORE 523168
hdb_town,Tampines,Tampines,Tampines
postal_code,520228,520505,523168
formatted_price,"$800,000","$685,000","$899,999"
SAI_Score,75.3,75.0,73.4
flat_type,5 ROOM,4 ROOM,4 ROOM
nearest_mrt_distance_m,247.6,218.8,274.5
num_mrt_within_1000m,3.0,2.0,3.0
nearest_clinic_distance_m,298.1,0.0,389.0


In [55]:
# Test Case 2: Filter by town, min_rooms, and budget
print("Test Case 2: Tampines, Min 3 rooms, Budget <= 600,000")
result_2 = final_output(final_df, town='Tampines', min_rooms=3, budget=600000, user_sliders=user_sliders)
result_2.T

Test Case 2: Tampines, Min 3 rooms, Budget <= 600,000


,10604,1151,8712
listing_url,https://www.propertyguru.com.sg/listing/hdb-fo...,https://www.propertyguru.com.sg/listing/hdb-fo...,https://www.propertyguru.com.sg/listing/hdb-fo...
onemap_full_address,406 TAMPINES STREET 41 SUN PLAZA GREEN SINGAPO...,419 TAMPINES STREET 41 SUN PLAZA GARDENS SINGA...,410 TAMPINES STREET 41 SUN PLAZA VIEW SINGAPOR...
hdb_town,Tampines,Tampines,Tampines
postal_code,520406,520419,520410
formatted_price,"$488,000","$599,000","$450,000"
SAI_Score,72.6,70.7,69.4
flat_type,3 ROOM,4 ROOM,3 ROOM
nearest_mrt_distance_m,403.0,471.6,515.9
num_mrt_within_1000m,2.0,2.0,2.0
nearest_clinic_distance_m,0.0,0.0,116.2


In [56]:
# Test Case 3: Filter by town, min_rooms, and budget
print("Test Case 3: Jurong East, Min 3 rooms, Budget <= 400,000")
result_3 = final_output(final_df, town='Jurong East', min_rooms=3, budget=400000, user_sliders=user_sliders)
result_3.T

Test Case 3: Jurong East, Min 3 rooms, Budget <= 400,000


,2375,1614,1424
listing_url,https://www.propertyguru.com.sg/listing/hdb-fo...,https://www.propertyguru.com.sg/listing/hdb-fo...,https://www.propertyguru.com.sg/listing/hdb-fo...
onemap_full_address,247 JURONG EAST STREET 24 SINGAPORE 600247,110 JURONG EAST STREET 13 HAPPY TALENT CHILDCA...,303 JURONG EAST STREET 32 HONG KAH EAST GARDEN...
hdb_town,Jurong East,Jurong East,Jurong East
postal_code,600247,600110,600303
formatted_price,"$388,000","$400,000","$398,000"
SAI_Score,71.2,70.4,70.3
flat_type,3 ROOM,3 ROOM,3 ROOM
nearest_mrt_distance_m,627.1,433.2,440.3
num_mrt_within_1000m,2.0,2.0,1.0
nearest_clinic_distance_m,65.7,281.2,0.0


In [57]:
final_df.to_csv('propertyguru_listings_final.csv', index=False)